# 18. Refit, validate, and prepare the final handoff

Owner: Jalil-g. Issue #18. Depends on completed #6, #7, #8, #9, #12, #16, #17.

## Goal

Run the fixed #16 neural blend plan, then refit the chosen model on all 10000 verified labels from raw files and export a checked test file. Nothing is uploaded; Jalil does that by hand.

## Rule (fixed before running)

**Planned vs actual pick.** The issue planned a classical candidate with hist threads 2. #17 changed that: its rule picked `hist50_old50` (augmented style .5, old soft text .5, NB transfer 0) with one thread everywhere. We use the #17 pick and its exact settings. NB transfer has weight 0, so it is not fit.

**Neural blend (#16 plan, written before #17 existed).** Inputs are the saved 8000-fit scores: #17 classical blend score and #16 ELECTRA margin. Each is standardized on eligible unlabeled rows (mean, population std). Score = (1 - w) * classical + w * ELECTRA for w in 0, .1, .25, .5, .75, 1. Objective is mean AUROC over original, remove blank lines, collapse whitespace, wrap80. Guardrail: original AUROC >= w=0 original - .002. Exact ties go to the smaller w. P(AI) = sigmoid(score).

**Caveat known up front.** Only frozen 2000 has 8000-fit scores for both models, so the weight is picked on frozen validation. That was the #16 plan, but it means the frozen numbers for the picked blend are selection numbers, not a clean confirmation. Frozen 2000 was already reused in #4, #8, #9, #13, #14, #15, #17.

**Full refit.** All 10000 labels are used only after the selection file is saved. Style HistGB fits in original `train.csv` order. Linear text models and ELECTRA fit in manifest order: training 8000 then validation 2000. Every reference statistic is recomputed on eligible unlabeled rows. One thread for sklearn. ELECTRA runs on MPS with the #16 settings and 3 epochs; MPS is not bitwise deterministic.

**Export.** `submissions/18_final_checks/<run>/submission.csv` with columns `id,label`, label = P(AI). Checked against sample IDs and recomputed by ID from shuffled text order.

## Setup

Helpers come from notebooks 10, 13, and 08 in helpers-only mode, so nothing there fits or writes. Arm code is copied from #17 and ELECTRA code from #16. A replay on the 8000 rows must match the #17 scores before the full refit. Runs in `envs/electra` because ELECTRA needs torch.

In [ ]:
import os, json, time, math, textwrap, warnings, platform, subprocess
from contextlib import contextmanager
from pathlib import Path
import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.special import expit, logit
from IPython import get_ipython
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from threadpoolctl import threadpool_limits
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
os.environ["EXPANDED_TEXT_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./10_expanded_text.ipynb")
os.environ["RICH_STYLE_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./13_rich_style.ipynb")
os.environ["ERROR_ANALYSIS_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./08_error_analysis.ipynb")
DIAG_CONFIG = {k: CONFIG[k] for k in ["measures", "bins", "threshold", "bootstrap", "cases_per_type", "case_order"]}
hf_logging.set_verbosity_error()

NAME = "18_final_checks"
COMMAND = "uv run --project envs/electra --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=7200 notebooks/18_final_checks.ipynb"
# Classical pick from #17. NB transfer weight is 0, so only two arms are fit.
STYLE_HIST_SETTINGS = {"max_iter": 250, "learning_rate": .06, "max_leaf_nodes": 15, "min_samples_leaf": 30,
                       "l2_regularization": 5., "early_stopping": False, "random_state": 42}
COPY_WEIGHT, STYLE_COPIES = .25, ["collapse"]
CLASSICAL = {"selected": "hist50_old50", "weights": [.5, .5, 0.]}
# Neural blend plan, copied from the #16 record and checked below.
NEURAL_WEIGHTS, NEURAL_SLACK = [0, .1, .25, .5, .75, 1], .002
VIEWS = ["original", "remove_blank", "collapse", "wrap80"]
ELECTRA_VIEW = {"original": "original", "remove_blank": "remove_blank_lines", "collapse": "collapse_whitespace", "wrap80": "wrap80"}
# ELECTRA settings from #16. 3 epochs was its internal pick.
MAX_TOKENS, BATCH, EVAL_BATCH, LR, WEIGHT_DECAY, WARMUP_FRACTION, SCHEDULE_EPOCHS, EPOCHS = 256, 16, 64, 3e-5, .01, .1, 3, 3
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
CKPT = ROOT / "artifacts" / "16_local_electra" / "checkpoint"
CHECKPOINT_FILES = ["config.json", "pytorch_model.bin", "tokenizer.json", "tokenizer_config.json", "vocab.txt"]
INPUT_RUNS = {"16": "20261110T222108811422Z", "17": "20261110T231659204936Z"}  # private 8000-fit score folders
CONFIG = {
    "planned_vs_actual": {"issue_plan": "classical candidate, hist threads 2, linear threads 1",
                          "actual": "#17 rule picked hist50_old50; one thread for every sklearn fit, same as #17"},
    "classical": CLASSICAL, "style": {"histgb": STYLE_HIST_SETTINGS, "copies": STYLE_COPIES, "copy_weight": COPY_WEIGHT,
                                      "score": "logit(clip(P(AI), 1e-7, 1-1e-7)), standardized on eligible unlabeled"},
    "old_soft": {"word": {"view": "word_char", "learner": "lr", "C": 16.}, "specialist": {"view": "casechar", "learner": "svm", "C": 1.},
                 "flag": "literal two consecutive newlines; active only when zero human fit rows have it", "offset": 2.},
    "neural_blend": {"weights": NEURAL_WEIGHTS, "score": "(1-w)*std(classical blend score) + w*std(ELECTRA margin), std on eligible unlabeled, population std",
                     "objective": "mean AUROC over original, remove_blank, collapse, wrap80", "guardrail": "original >= w0 original - .002",
                     "ties": "smaller w", "rows": "frozen 2000, 8000-fit scores from #16 and #17 (the only rows with both)",
                     "view_note": "classical wrap80 is #17 paragraph wrap; ELECTRA wrap80 is #16 line wrap; ELECTRA moves < 1e-6 under any view"},
    "full_refit": {"rows": 10000, "style_order": "original train.csv order", "linear_order": "manifest training then manifest validation",
                   "electra_order": "manifest training then manifest validation", "reference": "eligible unlabeled in split order",
                   "threads": "threadpool_limits(limits=1) for every sklearn fit and score", "electra_epochs": EPOCHS},
    "electra": {"max_tokens": MAX_TOKENS, "batch": BATCH, "eval_batch": EVAL_BATCH, "lr": LR, "weight_decay": WEIGHT_DECAY,
                "warmup_fraction": WARMUP_FRACTION, "schedule_epochs": SCHEDULE_EPOCHS, "device": DEVICE,
                "nondeterminism": "MPS kernels are not bitwise deterministic"},
    "export": {"path": "submissions/18_final_checks/<run>/submission.csv", "columns": ["id", "label"], "label": "P(AI), class 1",
               "order": "sample_submission order", "recompute": "shuffled test order, seed 42, matched by ID, max abs diff <= 1e-12"},
    "upload": "never automatic; Jalil uploads by hand",
}
START = time.perf_counter()
print("Device:", DEVICE, "| torch", torch.__version__)

## Checks

Notebook 02 checks data bytes and the frozen split. We check the dependency records, the #16 and #17 settings we copy, the private score files against their recorded hashes, and the pinned checkpoint files. The full 10000 rows must be exactly training plus validation.

In [ ]:
parts = load_partitions()
train, validation, reference = parts["train"], parts["validation"], parts["unlabeled"]
test, sample = read_raw("test.csv"), read_raw("sample_submission.csv")
validate_ids(test); validate_ids(sample)
require(len(test) == 3000 and set(test.id) == set(sample.id) and list(sample.columns) == ["id", "label"], "Sample and test IDs differ")
require(not set(test.id) & (set(train.id) | set(validation.id)), "Test overlaps labeled rows")
train_csv = read_raw("train.csv")  # original file order, for the style fit
manifest_full = pd.concat([train, validation], ignore_index=True)  # manifest order, for linear and ELECTRA fits
require(set(train_csv.id) == set(manifest_full.id) and len(train_csv) == 10000, "Full rows must be training plus validation")
require((train_csv.set_index("id").label.loc[manifest_full.id].to_numpy() == manifest_full.label.to_numpy()).all(), "Label mismatch")

def latest_completed(folder):
    rows = [(p, json.loads(p.read_text())) for p in sorted((ROOT / "results" / folder).glob("2*Z.json"))]
    rows = [(p, r) for p, r in rows if r.get("status") == "completed"]
    require(bool(rows), f"Completed {folder} record is required")
    return rows[-1]

deps = {k: latest_completed(f) for k, f in [("06", "06_learning_curve"), ("07", "07_pseudo_labels"), ("08", "08_error_analysis"),
        ("09", "09_linear_style"), ("12", "12_seed_stress"), ("16", "16_local_electra"), ("17", "17_final_blend")]}
dep16, dep17 = deps["16"][1], deps["17"][1]
require(dep17["metrics"]["selected"]["candidate"] == CLASSICAL["selected"] and dep17["metrics"]["selected"]["weights"] == CLASSICAL["weights"], "#17 pick changed")
require(dep17["config"]["style"]["histgb"] == STYLE_HIST_SETTINGS and dep17["config"]["style"]["copies"] == STYLE_COPIES
        and dep17["config"]["style"]["copy_weight"] == COPY_WEIGHT, "#17 style settings changed")
require(dep16["config"]["blend"]["neural_weights"] == NEURAL_WEIGHTS and dep16["metrics"]["selected_epochs"] == EPOCHS, "#16 plan changed")
t16 = dep16["config"]["train"]
require([t16[k] for k in ["max_tokens", "batch", "eval_batch", "lr", "weight_decay", "warmup_fraction", "schedule_epochs"]]
        == [MAX_TOKENS, BATCH, EVAL_BATCH, LR, WEIGHT_DECAY, WARMUP_FRACTION, SCHEDULE_EPOCHS], "#16 train settings changed")
require(dep16["config"]["checkpoint"]["files_sha256"] == {n: file_hash(CKPT / n) for n in CHECKPOINT_FILES}, "Checkpoint files changed")

# Private 8000-fit scores used only for the neural blend pick. Each must match the hash in its own record.
inputs = {"17_unlabeled": f"artifacts/17_final_blend/{INPUT_RUNS['17']}/components_unlabeled.csv",
          **{f"17_validation_{v}": f"artifacts/17_final_blend/{INPUT_RUNS['17']}/components_validation_{v}.csv" for v in VIEWS},
          "17_validation_probs": f"artifacts/17_final_blend/{INPUT_RUNS['17']}/blend_validation_original.csv",
          "16_unlabeled": f"artifacts/16_local_electra/{INPUT_RUNS['16']}/eligible_unlabeled_margins.csv",
          "16_validation": f"artifacts/16_local_electra/{INPUT_RUNS['16']}/frozen2000_margins.csv"}
for key, rel in inputs.items():
    record = dep17 if key.startswith("17") else dep16
    require(record["artifacts"].get(rel) == file_hash(ROOT / rel), f"Private input {rel} missing or changed")

CONFIG["dependencies"] = {k: {"record": str(p.relative_to(ROOT)), "record_sha256": file_hash(p)} for k, (p, _) in deps.items()}
CONFIG["private_inputs_sha256"] = {rel: file_hash(ROOT / rel) for rel in inputs.values()}
CONFIG["checkpoint_sha256"] = dep16["config"]["checkpoint"]["files_sha256"]
CONFIG["reference"] = {"rows": len(reference), "ordered_id_sha256": value_hash(reference.id.tolist())}
CONFIG["full_order_sha256"] = {"style": value_hash(train_csv.id.tolist()), "manifest": value_hash(manifest_full.id.tolist())}
CONFIG["helper_source_sha256"] = {n: value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks" / n).read_text())["cells"]])
                                  for n in ["08_error_analysis.ipynb", "10_expanded_text.ipynb", "13_rich_style.ipynb"]}
CONFIG["notebook_code_sha256"] = value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks/18_final_checks.ipynb").read_text())["cells"] if c["cell_type"] == "code"])
CONFIG["neural_lock"] = {"path": "envs/electra/uv.lock", "sha256": file_hash(ROOT / "envs/electra/uv.lock")}
CONFIG["neural_versions"] = {n: importlib.metadata.version(n) for n in ["torch", "transformers", "tokenizers", "safetensors", "cloudpickle", "threadpoolctl", "matplotlib"]}
chip = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip() if platform.system() == "Darwin" else platform.processor()
CONFIG["hardware"] = {"device": DEVICE, "chip": chip, "platform": platform.platform(), "torch_threads": torch.get_num_threads()}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT / "artifacts" / NAME / RUN_ID
private.mkdir(parents=True, exist_ok=False)
submission_dir = ROOT / "submissions" / NAME / RUN_ID
submission_dir.mkdir(parents=True, exist_ok=False)
print(f"Train {len(train)}, validation {len(validation)}, eligible unlabeled {len(reference)}, test {len(test)}")
print("Dependencies, settings, private inputs, and checkpoint hashes match.")

## Method

Style and old soft text arms are the #17 code. ELECTRA training is the #16 code with the same seed, schedule, and truncation. Every standardization uses eligible unlabeled rows only. The submission checker is tested on one valid file and eight broken ones.

In [ ]:
def literal_blank_flags(texts):
    return np.asarray(["\n\n" in str(t) for t in texts], dtype=bool)

def wrap_view(text, width):
    chunks = [c for c in PARAGRAPH_SPLIT.split(str(text)) if c.strip()]
    return "\n\n".join(textwrap.fill(c, width=width, break_long_words=False, break_on_hyphens=False) for c in chunks)

def stress_texts(texts, variant):
    if variant == "original": return [str(t) for t in texts]
    if variant == "remove_blank": return ["\n".join(line for line in str(t).split("\n") if line.strip()) for t in texts]
    if variant == "collapse": return [" ".join(str(t).split()) for t in texts]
    if variant == "wrap80": return [wrap_view(t, 80) for t in texts]
    raise ValueError("Unknown stress view")

def ref_stats(values):
    values = np.asarray(values, dtype=float)
    stats = {"mean": float(values.mean()), "std": float(values.std())}
    require(stats["std"] > 1e-12, "Degenerate reference")
    return stats

def standardize(values, stats):
    return (np.asarray(values, dtype=float) - stats["mean"]) / stats["std"]

class StyleArm:
    """#13 HistGB on rich-style features with #14 plus_collapse copies (#17 code)."""
    def fit(self, texts, labels, reference_texts):
        texts, labels = list(texts), np.asarray(labels)
        views = [texts] + [stress_texts(texts, v) for v in STYLE_COPIES]
        X, y = np.vstack([rich_style_features(v) for v in views]), np.concatenate([labels] * len(views))
        weights = np.concatenate([np.ones(len(labels))] + [np.full(len(labels), COPY_WEIGHT)] * len(STYLE_COPIES))
        self.model_ = HistGradientBoostingClassifier(**STYLE_HIST_SETTINGS).fit(X, y, sample_weight=weights)
        self.statistics_ = ref_stats(self.raw_margin(reference_texts))
        return self
    def raw_margin(self, texts): return logit(np.clip(class1_probabilities(self.model_, rich_style_features(list(texts))), 1e-7, 1 - 1e-7))
    def decision_function(self, texts): return standardize(self.raw_margin(texts), self.statistics_)

class SoftTextArm:
    """#11/#15 old soft text: word+char LR, no-blank char SVM, literal blank flag (#17 code)."""
    def fit(self, texts, labels, reference_texts):
        texts = pd.Series(list(texts)); labels = np.asarray(labels); keep = ~literal_blank_flags(texts)
        self.rule_active_ = not bool(literal_blank_flags(texts[labels == 0]).any())
        require(set(labels[keep]) == {0, 1}, "No-blank specialist needs both classes")
        self.word_ = make_expanded_model(CONFIG["old_soft"]["word"]).fit(texts, labels)
        self.specialist_ = make_expanded_model(CONFIG["old_soft"]["specialist"]).fit(texts[keep], labels[keep])
        reference_texts = pd.Series(list(reference_texts))
        self.word_stats_ = margin_reference(self.word_, reference_texts)
        self.specialist_stats_ = margin_reference(self.specialist_, reference_texts[~literal_blank_flags(reference_texts)])
        self.outer_ = ref_stats(self.raw_margin(reference_texts))
        self.fit_rows_, self.specialist_rows_ = len(texts), int(keep.sum())
        return self
    def raw_margin(self, texts):
        score = .5 * (standardized_margin(self.word_, texts, self.word_stats_) + standardized_margin(self.specialist_, texts, self.specialist_stats_))
        return score + 2. * literal_blank_flags(texts) * self.rule_active_
    def decision_function(self, texts): return standardize(self.raw_margin(texts), self.outer_)

def fit_classical(style_frame, linear_frame, reference_frame):
    seconds = {}
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        t = time.perf_counter(); style = StyleArm().fit(style_frame.text, style_frame.label, reference_frame.text); seconds["style"] = time.perf_counter() - t
        t = time.perf_counter(); old = SoftTextArm().fit(linear_frame.text, linear_frame.label, reference_frame.text); seconds["old_soft"] = time.perf_counter() - t
    return style, old, seconds

def classical_score(style, old, texts):
    with threadpool_limits(limits=1):
        return CLASSICAL["weights"][0] * style.decision_function(texts) + CLASSICAL["weights"][1] * old.decision_function(texts)

# ELECTRA, same code path as #16.
tokenizer = AutoTokenizer.from_pretrained(CKPT, local_files_only=True)
def encode(texts): return tokenizer(list(map(str, texts)), truncation=True, max_length=MAX_TOKENS)["input_ids"]
def to_batch(input_ids):
    batch = tokenizer.pad({"input_ids": input_ids}, return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def electra_margins(model, texts):
    model.eval(); ids, out = encode(texts), []
    for i in range(0, len(ids), EVAL_BATCH):
        logits = model(**to_batch(ids[i:i + EVAL_BATCH])).logits.float()
        out.append((logits[:, 1] - logits[:, 0]).cpu().numpy())
    values = np.concatenate(out).astype(float)
    require(values.shape == (len(texts),) and np.isfinite(values).all(), "Bad margins")
    return values

def train_electra(fit, epochs):
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2, local_files_only=True).to(DEVICE)
    ids, labels = encode(fit.text), torch.tensor(fit.label.to_numpy(), dtype=torch.long)
    steps_per_epoch = math.ceil(len(ids) / BATCH); total_steps = steps_per_epoch * SCHEDULE_EPOCHS
    warmup_steps = round(WARMUP_FRACTION * total_steps)
    decay = [p for n, p in model.named_parameters() if not (n.endswith("bias") or "LayerNorm" in n)]
    no_decay = [p for n, p in model.named_parameters() if n.endswith("bias") or "LayerNorm" in n]
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.}], lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    generator, history = torch.Generator().manual_seed(SEED), []
    for epoch in range(1, epochs + 1):
        model.train(); order, losses, start = torch.randperm(len(ids), generator=generator).tolist(), [], time.perf_counter()
        for i in range(0, len(order), BATCH):
            idx = order[i:i + BATCH]
            loss = model(**to_batch([ids[j] for j in idx]), labels=labels[idx].to(DEVICE)).loss
            loss.backward(); optimizer.step(); scheduler.step(); optimizer.zero_grad(); losses.append(loss.item())
        if DEVICE == "mps": torch.mps.synchronize()
        history.append({"epoch": epoch, "steps_done": epoch * steps_per_epoch, "total_schedule_steps": total_steps, "warmup_steps": warmup_steps,
                        "lr_end": scheduler.get_last_lr()[0], "mean_train_loss": float(np.mean(losses)), "train_seconds": time.perf_counter() - start})
        print({k: round(v, 6) if isinstance(v, float) else v for k, v in history[-1].items()})
    return model, history

def neural_grid(labels, classical, neural):
    """classical, neural: dict view -> standardized scores on the same rows."""
    rows = []
    for w in NEURAL_WEIGHTS:
        row = {"neural_weight": w, **{v: float(roc_auc_score(labels, (1 - w) * classical[v] + w * neural[v])) for v in VIEWS}}
        row["mean_auroc"] = float(np.mean([row[v] for v in VIEWS])); rows.append(row)
    floor = rows[0]["original"] - NEURAL_SLACK
    for row in rows: row["eligible"] = row["original"] >= floor
    best = max(r["mean_auroc"] for r in rows if r["eligible"])
    return rows, next(r for r in rows if r["eligible"] and r["mean_auroc"] == best)  # grid order = smaller w first

def calibration(labels, p, bins=10):
    labels, p = np.asarray(labels), np.asarray(p)
    index = np.minimum((p * bins).astype(int), bins - 1)
    table = [{"bin": f"[{k / bins:.1f},{(k + 1) / bins:.1f})", "n": int((index == k).sum()),
              "mean_p_ai": float(p[index == k].mean()) if (index == k).any() else None,
              "fraction_ai": float(labels[index == k].mean()) if (index == k).any() else None} for k in range(bins)]
    ece = sum(r["n"] * abs(r["mean_p_ai"] - r["fraction_ai"]) for r in table if r["n"]) / len(p)
    return {"brier": float(brier_score_loss(labels, p)), "log_loss": float(log_loss(labels, np.clip(p, 1e-15, 1 - 1e-15))),
            "ece_10_bins": float(ece), "bins": table}

def check_submission(path, sample_ids, expected):
    """Read the file back as text IDs. expected: dict id -> P(AI) recomputed separately. Raises on any problem."""
    frame = pd.read_csv(path, dtype={"id": "string"})
    require(list(frame.columns) == ["id", "label"], "Columns must be exactly id,label")
    require(frame.id.notna().all() and frame.id.is_unique, "Missing or duplicate IDs")
    require(set(frame.id) == set(sample_ids) and len(frame) == len(sample_ids), "IDs must match sample exactly")
    require(pd.api.types.is_numeric_dtype(frame.label), "Labels must be numeric")
    require(np.isfinite(frame.label).all() and frame.label.between(0, 1).all(), "Labels must be finite in [0, 1]")
    diff = np.abs(frame.label.to_numpy() - frame.id.map(expected).to_numpy(dtype=float))
    require(float(diff.max()) <= 1e-12, "Values do not match recomputed predictions by ID")
    return {"rows": len(frame), "max_abs_diff_vs_recompute": float(diff.max())}

@contextmanager
def guarded(stage):
    try:
        yield
    except Exception as exc:
        record_run(NAME, COMMAND, CONFIG, SEED, 0, 0, {"validation_auroc": None, "failed_stage": stage},
                   time.perf_counter() - START, artifacts=artifacts, status="failed", error_type=type(exc).__name__)
        raise RuntimeError(f"Stage {stage} failed. See the safe failed record.") from None

In [ ]:
# Synthetic checks: selection rule, tie order, guardrail, and the submission checker on valid and broken files.
y = np.array([0, 0, 1, 1])
flat = {v: np.array([0., 1., 2., 3.]) for v in VIEWS}
rows, pick = neural_grid(y, flat, flat)
require(pick["neural_weight"] == 0, "Exact tie must go to w=0")
weak = {**flat, "collapse": np.array([1., 0., 0.5, 2.])}
rows, pick = neural_grid(y, weak, {v: np.array([0., 1., 2., 3.]) for v in VIEWS})
require(pick["neural_weight"] > 0, "Robust neural score should win when classical breaks")
rows, pick = neural_grid(y, flat, {v: np.array([3., 2., 1., 0.]) for v in VIEWS})
require(pick["neural_weight"] == 0 and not rows[-1]["eligible"], "Guardrail must block a bad neural score")
cal = calibration([0, 1, 1, 0], [.05, .95, 1., .5])
require(cal["bins"][9]["n"] == 2 and cal["bins"][5]["n"] == 1, "Calibration bins failed")
require(stress_texts(["a\n \nb"], "remove_blank") == ["a\nb"] and stress_texts(["a  b\n\nc"], "collapse") == ["a b c"], "Stress fixture failed")

fixture_dir = private / "checker_fixtures"; fixture_dir.mkdir()
ids = [f"{n:05d}" for n in [1, 2, 3, 4]]  # leading zeros catch int parsing
p = np.array([.1, .2, .7, .9]); expected = dict(zip(ids, p))
good = pd.DataFrame({"id": ids, "label": p})
broken = {"wrong_columns": good.rename(columns={"label": "p_ai"}), "missing_row": good.iloc[:3],
          "extra_row": pd.concat([good, pd.DataFrame({"id": ["00009"], "label": [.5]})]), "duplicate_id": good.assign(id=[ids[0]] + ids[:3]),
          "nan_value": good.assign(label=[.1, np.nan, .7, .9]), "out_of_range": good.assign(label=[.1, .2, .7, 1.2]),
          "shifted_values": good.assign(label=np.roll(p, 1)), "wrong_class_column": good.assign(label=1 - p)}
good.to_csv(fixture_dir / "valid.csv", index=False)
require(check_submission(fixture_dir / "valid.csv", ids, expected)["max_abs_diff_vs_recompute"] == 0, "Valid fixture failed")
checker_fixtures = {"valid": "passed"}
for name, frame in broken.items():
    frame.to_csv(fixture_dir / f"{name}.csv", index=False)
    try:
        check_submission(fixture_dir / f"{name}.csv", ids, expected); checker_fixtures[name] = "accepted (bad)"
    except ValueError as exc:
        checker_fixtures[name] = f"rejected: {exc}"
require(all(v.startswith("rejected") for k, v in checker_fixtures.items() if k != "valid"), "Checker accepted a broken file")
display(pd.Series(checker_fixtures, name="checker").to_frame())

## Run

Step 1: neural blend pick on saved 8000-fit scores, saved before any new fit. Step 2: replay the classical arms on the same 8000 rows and match #17. Step 3: refit on all 10000 and score test. Step 4: write and check the submission. Any failure writes a safe failed record.

In [ ]:
plan_path = ROOT / "results" / NAME / f"{RUN_ID}-plan.json"
save_new_json(plan_path, {"config": CONFIG, "config_sha256": value_hash(CONFIG), "saved_before_fit": True, "split_sha256": parts["split"]["split_sha256"]})
artifacts, timings = [plan_path], {}
with guarded("neural_selection"):
    load = lambda key: pd.read_csv(ROOT / inputs[key], dtype={"id": "string"})
    c_ref = load("17_unlabeled"); e_ref = load("16_unlabeled")
    require(c_ref.id.tolist() == reference.id.tolist() and set(e_ref.id) == set(reference.id), "Reference rows differ")
    c_stats, e_stats = ref_stats(c_ref.blend_score), ref_stats(e_ref.margin_original)
    e_val = load("16_validation").set_index("id").loc[validation.id]
    classical_val, neural_val = {}, {}
    for v in VIEWS:
        frame = load(f"17_validation_{v}")
        require(frame.id.tolist() == validation.id.tolist(), "Validation order differs")
        classical_val[v] = standardize(frame.blend_score, c_stats)
        neural_val[v] = standardize(e_val[f"margin_{ELECTRA_VIEW[v]}"], e_stats)
    neural_rows, neural_pick = neural_grid(validation.label.to_numpy(), classical_val, neural_val)
    W = float(neural_pick["neural_weight"])
    selection_path = ROOT / "results" / NAME / f"{RUN_ID}-selection.json"
    save_new_json(selection_path, {"neural_grid": neural_rows, "selected": neural_pick, "classical": CLASSICAL,
        "reference_stats_8000_fit": {"classical": c_stats, "electra": e_stats}, "rows": "frozen 2000 (selection, reused validation)",
        "full_labels_used": False, "utc": datetime.now(timezone.utc).isoformat()})
    artifacts.append(selection_path)
print(f"Neural weight picked before any 10000 fit: w = {W}")
display(pd.DataFrame(neural_rows).set_index("neural_weight").round(6))

In [ ]:
# Replay: same code on the same 8000 rows must give the #17 scores. This checks the copied code, not a new score.
with guarded("replay_8000"):
    t = time.perf_counter()
    style8, old8, replay_fit_seconds = fit_classical(train, train, reference)
    replay = {"style_aug": style8.decision_function(validation.text), "old_soft": old8.decision_function(validation.text)}
    replay["blend_score"] = CLASSICAL["weights"][0] * replay["style_aug"] + CLASSICAL["weights"][1] * replay["old_soft"]
    saved17 = load("17_validation_original")
    replay_diff = {k: float(np.max(np.abs(replay[k] - saved17[k].to_numpy()))) for k in replay}
    require(max(replay_diff.values()) <= 1e-9, f"Replay differs from #17: {replay_diff}")
    timings["replay_8000_seconds"] = time.perf_counter() - t
    del style8, old8
print("Replay vs #17 max abs difference:", replay_diff)

In [ ]:
# Full refit on all 10000 verified labels. Selection is already saved.
with guarded("full_refit"):
    t = time.perf_counter()
    style, old, full_fit_seconds = fit_classical(train_csv, manifest_full, reference)
    timings["full_classical_fit_seconds"] = time.perf_counter() - t
    full_stats = {"classical": ref_stats(classical_score(style, old, reference.text))}
    electra, electra_history = None, []
    if W > 0:
        t = time.perf_counter()
        electra, electra_history = train_electra(manifest_full, EPOCHS)
        timings["full_electra_train_seconds"] = time.perf_counter() - t
        full_stats["electra"] = ref_stats(electra_margins(electra, reference.text))

    def final_scores(texts):
        texts = list(texts)
        score = standardize(classical_score(style, old, texts), full_stats["classical"])
        if W > 0:
            score = (1 - W) * score + W * standardize(electra_margins(electra, texts), full_stats["electra"])
        return expit(score)

    t = time.perf_counter()
    test_p = final_scores(test.text)
    # Recompute in a shuffled order and match by ID, to catch row shifts.
    order = np.random.default_rng(SEED).permutation(len(test))
    recomputed = dict(zip(test.id.iloc[order], final_scores(test.text.iloc[order])))
    shuffle_diff = float(np.max(np.abs(test_p - test.id.map(recomputed).to_numpy(dtype=float))))
    # In-sample direction check only: class 1 must mean AI. Not a held-out score.
    in_sample_auroc = float(roc_auc_score(manifest_full.label, final_scores(manifest_full.text)))
    require(in_sample_auroc > .9, "Class direction looks wrong")
    timings["full_scoring_seconds"] = time.perf_counter() - t
print(f"Full refit done. Shuffled-order max diff {shuffle_diff:.3g}. In-sample AUROC (not held out) {in_sample_auroc:.6f}")

In [ ]:
# Export in sample order and check the file as Kaggle would read it.
with guarded("export"):
    submission = pd.DataFrame({"id": sample.id}).merge(pd.DataFrame({"id": test.id, "label": test_p}), on="id", how="left", validate="one_to_one")
    submission_path = submission_dir / "submission.csv"
    submission.to_csv(submission_path, index=False)
    export_check = check_submission(submission_path, sample.id.tolist(), recomputed)
    export_check.update({"sha256": file_hash(submission_path), "shuffle_order_max_diff": shuffle_diff,
                         "label_min": float(submission.label.min()), "label_max": float(submission.label.max()),
                         "label_mean": float(submission.label.mean()), "share_above_0_5": float((submission.label >= .5).mean())})
    artifacts.append(submission_path)
    with (private / "classical_10000.pkl").open("wb") as handle:
        cloudpickle.dump({"style": style, "old_soft": old, "stats": full_stats, "weights": CLASSICAL["weights"], "neural_weight": W}, handle)
    artifacts.append(private / "classical_10000.pkl")
    if electra is not None:
        electra.save_pretrained(private / "electra_10000")
        artifacts += sorted(p for p in (private / "electra_10000").iterdir() if p.is_file())
print("Submission check:", export_check)

In [ ]:
# Diagnostics on the picked blend with 8000-fit scores on frozen 2000 (selection rows, so optimistic).
with guarded("diagnostics"):
    val_score = {v: (1 - W) * classical_val[v] + W * neural_val[v] for v in VIEWS}
    val_prob = {v: expit(val_score[v]) for v in VIEWS}
    confirmation = [{"view": v, **evaluate_predictions(validation, pd.DataFrame({"id": validation.id, "p_ai": val_prob[v]}))} for v in VIEWS]
    measures = {"train": measure(train), "validation": measure(validation)}
    edges = {column: quartile_edges(measures["train"][column]) for column in DIAG_CONFIG["measures"]}
    saved_val = pd.DataFrame({"id": validation.id, "p_ai": val_prob["original"]})
    overall = evaluate_predictions(validation[["id", "label"]], saved_val)
    overall["ci_low"], overall["ci_high"] = stratified_auc_interval(validation.label, val_prob["original"], DIAG_CONFIG["bootstrap"]["draws"], SEED)
    subgroups = subgroup_table(validation, saved_val, measures["validation"], edges, "validation", draws=DIAG_CONFIG["bootstrap"]["draws"], seed=SEED)
    cases, counts = confident_errors(validation, saved_val, measures["validation"], DIAG_CONFIG["threshold"], DIAG_CONFIG["cases_per_type"])
    path = private / "error_cases.csv"; cases.to_csv(path, index=False); artifacts.append(path)
    case_summary = cases.groupby("error")[["words", "line_breaks", "upper_per_letter", "punct_per_char"]].mean().reset_index()
    calibration_rows = {v: calibration(validation.label, val_prob[v]) for v in ["original", "collapse"]}
    # #17 probabilities, for the same views, so the change is visible.
    c17 = {v: float(roc_auc_score(validation.label, classical_val[v])) for v in VIEWS}

# Repeatability of notebooks 01-17, from the separate clean-kernel rerun summary (tools/compare_reruns.py).
rerun_files = sorted((ROOT / "results" / NAME).glob("*-rerun.json"))
rerun = json.loads(rerun_files[-1].read_text()) if rerun_files else None
timings["run_seconds"] = time.perf_counter() - START

## Results

Frozen 2000 rows picked the neural weight, so those numbers are selection numbers and optimistic. The 10000 refit has no held-out score. In-sample AUROC is only a direction check. None of this is a Kaggle score.

In [ ]:
print("Neural blend grid on frozen 2000 (8000-fit scores):")
display(pd.DataFrame(neural_rows).set_index("neural_weight").round(6))
print(f"Picked w = {W}. Floor on original: {neural_rows[0]['original'] - NEURAL_SLACK:.6f}")
print("Frozen 2000, picked blend vs #17 classical only:")
display(pd.DataFrame(confirmation).set_index("view").assign(classical_17=pd.Series(c17)).round(6))
print("Replay of #17 on 8000, max abs difference:", replay_diff)
print("Full refit state:", {"old_rule_active": old.rule_active_, "old_specialist_rows": old.specialist_rows_, "fit_rows": old.fit_rows_})
display(pd.DataFrame(electra_history).round(6) if electra_history else "ELECTRA not refit (w = 0)")
print("Notebook 08 diagnostics on the picked blend, frozen 2000:")
display(pd.DataFrame([overall]).round(6))
display(subgroups.drop(columns="partition").round(6))
print("Confusion at .5:", counts)
display(case_summary.round(4))
for view, cal in calibration_rows.items():
    print(f"Calibration on {view}: Brier {cal['brier']:.6f}, log loss {cal['log_loss']:.6f}, ECE10 {cal['ece_10_bins']:.6f}")
display(pd.DataFrame(calibration_rows["original"]["bins"]).round(4))
print("Submission:", export_check)
display(pd.Series(checker_fixtures, name="checker").to_frame())
if rerun:
    print(f"Clean-kernel rerun of notebooks 01-17 ({rerun['source']}):")
    display(pd.DataFrame(rerun["notebooks"]).set_index("notebook"))
else:
    print("No rerun summary found yet.")
display(pd.DataFrame([{"stage": k, "seconds": v} for k, v in {**{f"full_fit_{k}": s for k, s in full_fit_seconds.items()},
        **{f"replay_fit_{k}": s for k, s in replay_fit_seconds.items()}, **timings}.items()]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
grid = pd.DataFrame(neural_rows)
for v in VIEWS + ["mean_auroc"]:
    axes[0].plot(grid.neural_weight, grid[v], marker="o", label=v)
axes[0].axvline(W, color="gray", ls="--", lw=1)
axes[0].set(title="Frozen 2000 AUROC by neural weight", xlabel="ELECTRA weight w", ylabel="AUROC"); axes[0].legend(fontsize=8)
bins = pd.DataFrame(calibration_rows["original"]["bins"]).dropna()
axes[1].plot([0, 1], [0, 1], color="gray", linestyle=":"); axes[1].plot(bins.mean_p_ai, bins.fraction_ai, marker="o")
axes[1].set(title="Frozen 2000 reliability, picked blend", xlabel="mean P(AI) in bin", ylabel="fraction AI")
fig.tight_layout(); plot_path = ROOT / "results" / NAME / f"{RUN_ID}-final.png"; fig.savefig(plot_path, dpi=130); artifacts.append(plot_path); plt.show()

## Takeaway

- Tried: TAKEAWAY_TRIED
- Found: TAKEAWAY_FOUND
- Next: TAKEAWAY_NEXT

## Save

The safe record keeps the neural grid, replay check, refit state, submission checks, diagnostics, calibration, rerun summary, and timings. Models, error cases, and the submission stay in ignored `artifacts/` and `submissions/`. The 10000 refit has `validation_auroc` null and links the 8000-fit frozen evidence. Nothing is uploaded.

In [ ]:
metrics = {"validation_auroc": None, "neural_grid_frozen2000": neural_rows, "neural_selected": neural_pick,
    "frozen2000_picked_blend": confirmation, "frozen2000_classical_17": c17, "replay_max_abs_diff_vs_17": replay_diff,
    "full_refit": {"old_rule_active": old.rule_active_, "old_specialist_rows": old.specialist_rows_, "reference_stats": full_stats,
                   "style_stats": style.statistics_, "old_outer": old.outer_, "electra_history": electra_history,
                   "in_sample_auroc_not_heldout": in_sample_auroc},
    "submission": export_check, "checker_fixtures": checker_fixtures,
    "diagnostics": {"overall": overall, "edges": {c: [None if not np.isfinite(x) else float(x) for x in b] for c, b in edges.items()},
                    "subgroups": json.loads(subgroups.to_json(orient="records")), "confusion_at_threshold": counts,
                    "error_case_mean_measures": json.loads(case_summary.to_json(orient="records"))},
    "calibration": calibration_rows, "rerun_summary": {"file": str(rerun_files[-1].relative_to(ROOT)), "sha256": file_hash(rerun_files[-1])} if rerun else None,
    "fit_seconds": {"replay": replay_fit_seconds, "full": full_fit_seconds}, "timings": timings,
    "kaggle": {"uploaded_by_notebook": False, "status": "pending manual upload by Jalil", "public_score": None, "private_score": None}}
heldout = {"rows": "frozen 2000, fit on 8000", "selection_record": str(selection_path.relative_to(ROOT)),
           "classical_record": CONFIG["dependencies"]["17"], "electra_record": CONFIG["dependencies"]["16"],
           "picked_blend_original_auroc": confirmation[0]["auroc"], "note": "neural weight was picked on these rows, so this is optimistic"}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, 10000, 0, metrics, time.perf_counter() - START, artifacts=artifacts, heldout_confirmation=heldout)
print(f"Saved {record_path.relative_to(ROOT)}")